# 잠긴 종 · 한국어 룰북 RAG

수업에서 배운 Hugging Face 모델 실행과 PDF 기반 RAG를 팀의 D&D 베이직 룰 한국어판에 적용합니다.

**Colab 실행 방법**  
런타임 유형을 **GPU(T4 또는 L4)**로 선택하고 위에서 아래로 실행합니다. 첫 실행에는 Qwen3.5-4B와 임베딩 모델을 내려받습니다. 팀 룰북은 [한국어판 Rules.pdf](https://github.com/TRPG-masterbot/data/blob/93130b452d4351f0b051f1478a7e582e4156c549/Rules.pdf)를 사용합니다.

**실행 흐름**

    GPU 확인 → Qwen 로드와 짧은 생성 테스트
    한국어 PDF → 페이지와 본문 → 검색용 조각 → 임베딩 → Chroma
    질문 → 의미·키워드 검색 → 후보 구절 → Qwen 답변과 PDF 쪽수

10월 15일 Hugging Face 실습의 토크나이저·생성 흐름과 10월 27일 RAG 실습의 PDF 추출·Chroma·BM25/RRF 흐름을 따릅니다. 임베딩과 답변 생성에 쓰는 모델을 각각 표시합니다.

## 1. Colab GPU 확인

PyTorch에서 CUDA를 사용할 수 있는지 확인합니다. Qwen 생성은 GPU 런타임에서 실행합니다. CPU에서도 PDF 추출과 검색은 따라갈 수 있습니다.

In [ ]:
import torch

gpu_available = torch.cuda.is_available()

if gpu_available:
    print("CUDA 사용 가능")
else:
    print("CUDA 사용 불가 · 룰북 검색은 CPU에서 실행할 수 있습니다.")

## 2. 실습 라이브러리 설치

Transformers와 Accelerate는 Qwen 실행에 씁니다. PyMuPDF는 PDF를 읽고, Sentence Transformers는 임베딩을 만들며, Chroma와 BM25는 관련 구절을 검색합니다.

In [ ]:
%pip -q install "transformers>=5.3,<6" "accelerate>=1,<2" "pymupdf>=1.24,<2" "sentence-transformers>=5,<6" "chromadb>=1,<2" "rank-bm25==0.2.2"

## 3. Qwen과 토크나이저 불러오기

토크나이저는 질문을 모델이 읽는 숫자 ID로 바꿉니다. Qwen3.5-4B는 마지막 단계에서 룰북 근거를 읽고 답을 씁니다. GPU가 지원하는 데이터 형식을 선택해 가중치를 올립니다.

In [ ]:
from transformers import AutoTokenizer, Qwen3_5ForConditionalGeneration

qwen_id = "Qwen/Qwen3.5-4B"
qwen_revision = "851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a"
qwen_model = None

if gpu_available:
    qwen_tokenizer = AutoTokenizer.from_pretrained(qwen_id, revision=qwen_revision)
    qwen_dtype = (
        torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False)
        else torch.float16
    )
    qwen_model = Qwen3_5ForConditionalGeneration.from_pretrained(
        qwen_id, revision=qwen_revision, dtype=qwen_dtype, device_map="auto",
    )
    qwen_model.eval()
    print(f"{qwen_id} 준비 완료 · {qwen_model.device} · {qwen_dtype}")

## 4. 짧은 질문으로 Qwen 생성 확인

대화를 채팅 형식으로 바꿔 모델에 주고, 입력 뒤에 새로 생성된 토큰만 글로 되돌립니다. 아래 인사는 모델이 응답하는지 보는 테스트입니다. 같은 함수를 마지막 룰북 답변에서도 사용합니다.

In [ ]:
def generate_answer(messages, max_new_tokens=96):
    inputs = qwen_tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
        enable_thinking=False,
    ).to(qwen_model.device)

    with torch.inference_mode():
        output = qwen_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

    input_length = inputs["input_ids"].shape[-1]
    return qwen_tokenizer.decode(output[0, input_length:], skip_special_tokens=True).strip()


if qwen_model is not None:
    test_messages = [{"role": "user", "content": "한국어로 짧게 인사해 주세요."}]
    print("모델 응답", generate_answer(test_messages, max_new_tokens=32))

## 5. 한국어 룰북 가져오기와 페이지별 텍스트 추출

작업 폴더의 Rules.pdf를 쓰거나 팀 GitHub의 고정 버전에서 내려받습니다. 파일 지문으로 판본을 확인한 뒤, PyMuPDF로 페이지마다 글을 읽고 쪽수를 함께 저장합니다.

In [ ]:
from hashlib import sha256
import os
from pathlib import Path
from urllib.error import URLError
from urllib.request import urlopen

revision = "93130b452d4351f0b051f1478a7e582e4156c549"
pdf_url = f"https://raw.githubusercontent.com/TRPG-masterbot/data/{revision}/Rules.pdf"
expected_sha256 = "d00d1081a11d7f988f58769f0154aadcea0d32d8dbd04d31cef16980f57be985"
pdf_path = Path(os.environ.get("DROWNED_BELL_RULES_PDF", "Rules.pdf"))

if pdf_path.is_file():
    pdf_bytes = pdf_path.read_bytes()
else:
    if "DROWNED_BELL_RULES_PDF" in os.environ:
        raise FileNotFoundError(f"지정한 한국어 룰북 파일을 찾을 수 없습니다: {pdf_path}")
    try:
        with urlopen(pdf_url, timeout=30) as response:
            pdf_bytes = response.read()
    except URLError as error:
        raise RuntimeError("팀 GitHub에서 Rules.pdf를 받아 작업 폴더에 넣어 주세요.") from error

if sha256(pdf_bytes).hexdigest() != expected_sha256:
    raise ValueError("팀의 베이직 룰 한국어판 Rules.pdf와 파일 내용이 다릅니다.")
if not pdf_path.exists():
    pdf_path.write_bytes(pdf_bytes)

print(f"{pdf_path.name} · 한국어판 고정 버전 확인")

import pymupdf

pages = []
with pymupdf.open(stream=pdf_bytes, filetype="pdf") as pdf:
    print("전체 PDF 쪽수", len(pdf))
    for page_no, page in enumerate(pdf, start=1):
        text = page.get_text("text").strip()
        if text:
            pages.append({"page": page_no, "text": text})

print("텍스트를 읽은 쪽수", len(pages))

## 6. 임베딩 모델과 입력 길이 확인

MiniLM은 글을 비교할 수 있는 숫자 배열로 바꿉니다. 문서와 질문 모두 같은 임베딩 모델을 사용합니다. GPU 메모리가 충분하면 임베딩도 GPU에서 계산하고, 메모리가 부족하면 CPU를 사용합니다.

In [ ]:
# 3. 임베딩 모델과 토큰 한도 확인
import torch
from sentence_transformers import SentenceTransformer

model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
free_gpu_bytes = torch.cuda.mem_get_info()[0] if gpu_available else 0
embedding_device = "cuda" if free_gpu_bytes >= 2 * 1024**3 else "cpu"
model_revision = "e8f8c211226b894fcb81acc59f3b34ba3efd5f42"
embedder = SentenceTransformer(model_name, revision=model_revision, device=embedding_device)
embedding_tokenizer = embedder.tokenizer
token_limit = embedder.max_seq_length
print(f"임베딩 장치: {embedding_device} | 입력 한도: {token_limit}토큰")

## 7. 문장을 검색용 조각으로 나누기

페이지를 문장과 목록 단위로 나누고, 임베딩 모델의 토큰 한도 안에서 묶습니다. 긴 문장은 검색용으로 한 번 더 나누고 원래 조각 번호를 남깁니다. 이 번호가 있어야 검색 결과를 원문과 쪽수로 연결할 수 있습니다.

In [ ]:
# PDF를 문장·목록 단위로 묶습니다.
import re


def count_tokens(text):
    return len(embedding_tokenizer(text, truncation=False, verbose=False)["input_ids"])


def sentence_units(text):
    boundary = r"(?<=[.!?。！？])\s+|\n[ \t]*(?=[•●▪])|\n[ \t]*\n"
    return [" ".join(unit.split()) for unit in re.split(boundary, text) if unit.strip()]


def split_page(text):
    window = []
    for unit in sentence_units(text):
        if count_tokens(unit) > token_limit:
            if window:
                yield " ".join(window)
                window = []
            yield unit
            continue

        if window and count_tokens(" ".join(window + [unit])) > token_limit:
            yield " ".join(window)
            last_sentence = window[-1]
            window = [last_sentence] if count_tokens(f"{last_sentence} {unit}") <= token_limit else []
        window.append(unit)

    if window:
        yield " ".join(window)


def split_for_model(text, tokenizer, max_length, overlap_tokens=24):
    """긴 원문을 MiniLM이 읽을 수 있는 길이로 나눈다."""
    def length(s):
        return len(tokenizer(s, add_special_tokens=True, truncation=False, verbose=False)['input_ids'])
    if length(text) <= max_length:
        return [text]
    if not getattr(tokenizer, 'is_fast', False):
        raise ValueError('긴 조각 재분할에는 fast tokenizer가 필요합니다.')
    special = tokenizer.num_special_tokens_to_add(pair=False)
    budget = max_length - special
    if budget < 2:
        raise ValueError('모델 토큰 한도가 너무 작습니다.')
    offsets = tokenizer(text, add_special_tokens=False, truncation=False, verbose=False,
                        return_offsets_mapping=True)['offset_mapping']
    offsets = [(a, b) for a, b in offsets if b > a]
    pieces, start = [], 0
    while start < len(offsets):
        end = min(start + budget, len(offsets))
        # 잘라낸 경계에서 다시 토큰화하면 길이가 달라질 수 있으므로 재확인합니다.
        while end > start:
            left = 0 if start == 0 else offsets[start][0]
            right = len(text) if end == len(offsets) else offsets[end][0]
            part = text[left:right]
            if length(part) <= max_length:
                break
            end -= 1
        if end <= start:
            raise ValueError('한 토큰의 원문조차 모델 길이 안에 넣을 수 없습니다.')
        if part.strip():
            pieces.append(part)
        if end == len(offsets):
            break
        start = max(start + 1, end - min(overlap_tokens, budget // 4))
    assert pieces and all(length(p) <= max_length for p in pieces)
    return pieces


chunks = [
    {"page": page["page"], "text": part}
    for page in pages
    for part in split_page(page["text"])
]
texts = [chunk["text"] for chunk in chunks]

# 한도를 넘는 원문은 여러 임베딩에 연결하고 검색 결과에서는 한 번만 보여준다.
index_rows = [
    (position, part)
    for position, text in enumerate(texts)
    for part in split_for_model(text, embedding_tokenizer, token_limit)
]
index_texts = [part for _, part in index_rows]
max_tokens = max(map(count_tokens, index_texts))
assert max_tokens <= token_limit



print(f"원문 조각 {len(chunks)}개 · 임베딩 입력 {len(index_rows)}개")
print(f"가장 긴 임베딩 입력 {max_tokens}/{token_limit}토큰")

## 8. 룰북 임베딩을 만들어 Chroma에 넣기

embedder.encode()가 각 조각을 벡터로 바꿉니다. Chroma에 벡터, 검색용 문장, 원문 조각 번호와 쪽수를 넣습니다. 이 컬렉션은 노트북 실행 중에 사용합니다.

In [ ]:
# 각 문장 조각을 임베딩 벡터로 바꾸고 Chroma에 등록합니다.
import chromadb
from uuid import uuid4

vectors = embedder.encode(index_texts, normalize_embeddings=True, show_progress_bar=False)
client = chromadb.EphemeralClient(chromadb.config.Settings(anonymized_telemetry=False))
rules = client.create_collection(
    f"drowned_bell_rules_{uuid4().hex[:8]}",
    embedding_function=None,
    configuration={"hnsw": {"space": "cosine"}},
)
rules.add(
    ids=[f"part-{i}" for i in range(len(index_rows))],
    embeddings=vectors.tolist(),
    documents=index_texts,
    metadatas=[{"position": position, "page": chunks[position]["page"]} for position, _ in index_rows],
)
print("저장한 검색 벡터:", rules.count())

## 9. 의미 검색으로 관련 구절 찾기

질문도 벡터로 바꾸고 rules.query()에서 가까운 벡터를 찾습니다. 여러 벡터가 같은 원문에서 나왔다면 결과에는 그 원문을 한 번만 담습니다.

In [ ]:
# 가까운 벡터를 원문과 PDF 쪽수로 연결합니다.
def search_rules(question, top_k=3):
    query_vector = embedder.encode([question], normalize_embeddings=True).tolist()
    # 하나의 원문에 연결된 여러 벡터가 상위에 와도 필요한 원문 수를 확보한다.
    extra_vectors = len(index_rows) - len(chunks)
    matches = rules.query(
        query_embeddings=query_vector,
        n_results=min(top_k + extra_vectors, rules.count()),
        include=["metadatas", "distances"],
    )

    ranked = {}
    for meta, distance in zip(matches["metadatas"][0], matches["distances"][0]):
        ranked.setdefault(meta["position"], distance)
        if len(ranked) == top_k:
            break

    return {
        "ids": [[f"rule-{i}" for i in ranked]],
        "documents": [[chunks[i]["text"] for i in ranked]],
        "metadatas": [[{"file": pdf_path.name, "page": chunks[i]["page"]} for i in ranked]],
        "distances": [list(ranked.values())],
    }


def find_rules(question):
    result = search_rules(question)
    print(f"\n질문: {question}")
    for text, meta, distance in zip(
        result["documents"][0], result["metadatas"][0], result["distances"][0]
    ):
        print(f"\n{meta['file']} {meta['page']}쪽 | 거리 {distance:.3f}")
        print(text)


DEMO_QUESTION = "공격 굴림 결과가 상대 방어도와 같으면 명중해?"
find_rules(DEMO_QUESTION)

## 10. BM25 키워드 검색과 RRF로 순위 합치기

BM25는 질문에 나온 글자 조각으로 후보를 찾습니다. RRF는 의미 검색과 BM25에서 각각 몇 위였는지로 점수를 합칩니다. 두 검색에서 모두 높은 문단은 위로 올라갑니다.

In [ ]:
import re
from rank_bm25 import BM25Okapi


def keyword_tokens(text):
    tokens = []
    for word in re.findall(r"[가-힣]+|[a-z0-9]+", text.casefold()):
        if "가" <= word[0] <= "힣":
            tokens.extend(word[i:i + 2] for i in range(len(word) - 1))
        else:
            tokens.append(word)
    return tokens


keyword_index = BM25Okapi([keyword_tokens(text) for text in texts])
chunk_positions = {f"rule-{i}": i for i in range(len(chunks))}


def search_hybrid(question):
    semantic_ids = search_rules(question, top_k=10)["ids"][0]
    scores = keyword_index.get_scores(keyword_tokens(question))
    keyword_positions = [i for i, score in enumerate(scores) if score > 0]
    keyword_positions.sort(key=lambda i: scores[i], reverse=True)
    keyword_ids = [f"rule-{i}" for i in keyword_positions[:10]]

    combined = {}
    for ranked_ids in (semantic_ids, keyword_ids):
        for rank, chunk_id in enumerate(ranked_ids, start=1):
            combined[chunk_id] = combined.get(chunk_id, 0) + 1 / (60 + rank)

    top_ids = sorted(combined, key=combined.get, reverse=True)[:3]
    selected = [chunks[chunk_positions[chunk_id]] for chunk_id in top_ids]
    return {
        "ids": [top_ids],
        "documents": [[chunk["text"] for chunk in selected]],
        "metadatas": [[{"file": pdf_path.name, "page": chunk["page"]} for chunk in selected]],
    }

## 11. 같은 10문항으로 검색 결과 비교

질문마다 정해 둔 PDF 쪽수와 근거 문구를 확인합니다. 두 검색 방식에 같은 질문을 넣고, 맞는 근거가 첫 번째나 상위 세 결과 안에 있는지 셉니다.

In [ ]:
# 질문, 근거 쪽수, 해당 근거에서 확인할 문구
evaluation_questions = [
    ("공격 굴림 결과가 상대 방어도와 같으면 명중해?", 75,
     "굴림의 결과가 목표의 방어도(AC) 이상이라면 공격은 명중한 것입니다."),
    ("짧은 휴식에서 회복 주사위를 어떻게 써?", 69,
     "HD 주사위 하나를 굴릴 때마다 여기에 자신의 건강 수정치를 더하며"),
    ("붙잡힌 상태에서 이동할 수 있어?", 170,
     "붙잡힌 상태의 크리쳐는 이동속도가 0이 됩니다."),
    ("이점이 있으면 두 주사위 중 어떤 걸 골라?", 59,
     "이점인 경우 둘 중 높은 값을 사용"),
    ("능력 판정에서 굴림 결과가 난이도와 같아도 성공이야?", 60,
     "결과값이 DC보다 높거나 같다면 판정은 성공한 것입니다."),
    ("적의 기회공격을 피하면서 물러나려면 어떤 행동을 써?", 74,
     "퇴각 행동을 사용하면, 당신의 이동은 그 턴이 끝날 때까지 기회공격을 유발하지 않습니다."),
    ("절반 엄폐는 방어도에 보너스를 얼마나 줘?", 76,
     "절반 엄폐를 받는 대상은 AC와 민첩 내성 굴림에 +2 보너스를 받습니다."),
    ("치명타가 나면 피해 주사위를 몇 번 굴려?", 77,
     "공격의 피해 주사위를 두 번 굴린 다음"),
    ("죽음 내성 굴림은 몇 이상이면 성공이야?", 78,
     "결과가 10 이상이라면, 당신은 성공한 것입니다."),
    ("1레벨 마법 화살을 쓰면 다트가 몇 개 나와?", 89,
     "빛나는 3개의 다트를 만들어 냅니다."),
]

def normalize_text(text):
    return "".join(text.split()).casefold()


def evaluate_search(search):
    first_hits = 0
    top3_hits = 0

    for question, expected_page, evidence in evaluation_questions:
        result = search(question)
        found_pages = [meta["page"] for meta in result["metadatas"][0]]
        expected_text = normalize_text(evidence)

        matched_rank = None
        for rank, (page, text) in enumerate(zip(found_pages, result["documents"][0]), 1):
            if page == expected_page and expected_text in normalize_text(text):
                matched_rank = rank
                break

        first_hits += matched_rank == 1
        top3_hits += matched_rank is not None
        rank_label = f"{matched_rank}번째 결과에서 근거를 찾음" if matched_rank else "근거를 찾지 못함"
        if matched_rank is None and expected_page in found_pages:
            rank_label = "맞는 쪽을 찾았지만 답이 되는 문장은 빠짐"
        print(f"{question} → {rank_label} (기준 {expected_page}쪽 / 검색 {found_pages})")

    total = len(evaluation_questions)
    print(f"첫 번째 결과에서 근거를 찾은 질문: {total}개 중 {first_hits}개")
    print(f"첫 세 결과 안에서 근거를 찾은 질문: {total}개 중 {top3_hits}개")


print("[의미 검색]")
evaluate_search(search_rules)

print("\n[의미 + 키워드 검색]")
evaluate_search(search_hybrid)

## 12. Qwen으로 근거 선택과 한국어 답변

검색 상위 조각을 문장 후보로 나누고, Qwen으로 질문에 직접 답하는 근거 번호를 먼저 선택합니다. 선택한 원문 한 문장을 다시 Qwen에 줘서 한국어로 답합니다. 출처에는 실제 PDF 쪽수를 표시합니다.

In [ ]:
def evidence_candidates(question, result):
    # 검색에 나온 쪽에서 키워드가 겹치는 다른 조각을 하나씩 추가한다.
    positions = [chunk_positions[chunk_id] for chunk_id in result["ids"][0]]
    pages_to_check = dict.fromkeys(chunks[i]["page"] for i in positions)
    scores = keyword_index.get_scores(keyword_tokens(question))

    for page in pages_to_check:
        best = max(
            (i for i, chunk in enumerate(chunks)
             if chunk["page"] == page and i not in positions and scores[i] > 0),
            key=lambda i: scores[i],
            default=None,
        )
        if best is not None:
            positions.append(best)

    return list(dict.fromkeys(
        (chunks[i]["page"], sentence)
        for i in positions
        for sentence in sentence_units(chunks[i]["text"])
    ))


def select_evidence(question, candidates):
    if not candidates:
        return None

    numbered = "\n".join(
        f"{i}. [{page}쪽] {sentence}"
        for i, (page, sentence) in enumerate(candidates, start=1)
    )
    messages = [
        {
            "role": "system",
            "content": (
                "질문에 직접 답하는 룰북 문장 하나를 골라 숫자만 출력하세요. "
                "같은 주제를 언급해도 질문에 답하지 않으면 제외하세요. "
                "질문에서 행동하는 사람이 누구인지 확인하고, 문장에서 그 사람에게 적용되는 규칙만 고르세요. "
                "직접 답하는 문장이 없으면 0을 출력하세요."
            ),
        },
        {"role": "user", "content": f"질문: {question}\n\n룰북 후보:\n{numbered}"},
    ]
    answer = generate_answer(messages, max_new_tokens=12)
    if re.fullmatch(r"[0-9]{1,3}", answer):
        index = int(answer)
        if 1 <= index <= len(candidates):
            return candidates[index - 1]
    return None


def answer_rule(question):
    result = search_hybrid(question)
    selected = select_evidence(question, evidence_candidates(question, result))
    if selected is None:
        return None

    page, excerpt = selected
    messages = [
        {
            "role": "system",
            "content": (
                "사용자가 제공한 한국어 룰북 원문 한 문장에 근거해 질문에 답하세요. "
                "원문의 뜻, 수치, 적용 대상을 유지하세요. "
                "비교 규칙에서 이상은 기준을 포함해 그보다 큰 값이고, 이하는 기준을 포함해 그보다 작은 값입니다. "
                "질문에 직접 답하는 한국어 한 문장만 쓰세요."
            ),
        },
        {"role": "user", "content": f"룰북 원문: {excerpt}\n질문: {question}"},
    ]
    answer = " ".join(generate_answer(messages, max_new_tokens=70).split())
    if not answer:
        return None
    return answer, page, excerpt


if qwen_model is not None:
    print("질문", DEMO_QUESTION)
    result = answer_rule(DEMO_QUESTION)
    if result is None:
        print("검색 후보에서 답의 근거를 확인하기 어렵습니다.")
    else:
        answer, page, excerpt = result
        print(f"답변 {answer}")
        print(f"출처 D&D 베이직 룰 한국어판 {page}쪽")
        print("사용한 구절", excerpt)
else:
    print("Qwen 답변은 Colab GPU 런타임에서 실행합니다.")

## 화면에서 확인할 것

- 4단계는 모델 실행, 9단계는 검색, 12단계는 검색한 구절을 바탕으로 한 답변을 보여줍니다.
- 11단계는 **검색 근거**를 평가합니다. 12단계에서 생성한 답과 사용한 원문은 함께 확인합니다.
- 이 노트북은 수업 방식으로 구성한 룰북 실험입니다. 같은 패키지의 로컬 게임에서는 가벼운 글자 검색으로 근거를 찾아 로컬 Qwen에 전달합니다.